In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from scipy.stats import skew, kurtosis
from pathlib import Path

In [3]:
CACHE_DIR = Path("lightkurve_cache")

N_TOP_PLOTS = 5


In [4]:
def extract_features(flux,gap_mask):
    valid_flux = flux[gap_mask]

    if len(valid_flux) < 10:
        return None

    features = {
        "std" : np.std(valid_flux),
        "ptp" : np.ptp(valid_flux),
        "roughness" : np.mean(np.abs(np.diff(valid_flux))),
        "p5": np.percentile(valid_flux, 5),
        "p95": np.percentile(valid_flux, 95),
        "skewness" : skew(valid_flux),
        "kurtosis" : kurtosis(valid_flux)
    }

    return features

In [ ]:
def build_feature_dataframe():

    rows = []

    for cache_file in CACHE_DIR.glob("*.npy"):
        data = np.load(cache_file, allow_pickle=True).item()

        features = extract_features(data["flux"],data["gap_mask"])

        if features is None:
            print(f" Skipping {cache_file.name} (too few valid points)")
            continue

        features["tic_id"] = data["tic_id"]
        features["sector"] = data["sector"]
        features["cache_file"] = cache_file.name

        rows.append(features)

    return pd.DataFrame(rows)

In [ ]:
df = build_feature_dataframe()

df



,std,ptp,roughness,p5,p95,skewness,kurtosis,tic_id,sector,cache_file
0,0.000432,0.003152,0.000372,0.999287,1.000740,0.052321,0.232311,140997242,1,TIC140997242_sector1.npy
1,0.000443,0.003579,0.000380,0.999304,1.000722,-0.042450,0.561289,140997242,11,TIC140997242_sector11.npy
2,0.000476,0.003424,0.000411,0.999220,1.000762,0.016457,0.050178,140997242,12,TIC140997242_sector12.npy
3,0.000443,0.003529,0.000401,0.999263,1.000705,-0.020846,0.362420,140997242,13,TIC140997242_sector13.npy
4,0.000430,0.003249,0.000372,0.999299,1.000719,0.181831,0.417395,140997242,2,TIC140997242_sector2.npy
...,...,...,...,...,...,...,...,...,...,...
4015,0.002051,0.014049,0.002009,0.996713,1.003284,-0.031522,-0.126287,41173537,94,TIC41173537_sector94.npy
4016,0.001851,0.012375,0.001865,0.996985,1.002986,0.063659,0.119107,41173537,95,TIC41173537_sector95.npy
4017,0.002227,0.014449,0.002036,0.996300,1.003523,-0.105215,-0.212057,41173537,96,TIC41173537_sector96.npy
4018,0.002224,0.015413,0.002106,0.996300,1.003464,0.056617,-0.106107,41173537,97,TIC41173537_sector97.npy


In [7]:
print(df["tic_id"].nunique())

93
